## Preprocessing

## Libraries

In [2]:
pip install Librosa

   ---------------------------------------- 0.0/260.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/260.7 kB ? eta -:--:--
   ------ --------------------------------- 41.0/260.7 kB 1.9 MB/s eta 0:00:01
   ------ --------------------------------- 41.0/260.7 kB 1.9 MB/s eta 0:00:01
   ------------ -------------------------- 81.9/260.7 kB 657.6 kB/s eta 0:00:01
   ------------- ------------------------- 92.2/260.7 kB 525.1 kB/s eta 0:00:01
   ---------------- --------------------- 112.6/260.7 kB 504.4 kB/s eta 0:00:01
   ---------------- --------------------- 112.6/260.7 kB 504.4 kB/s eta 0:00:01
   -------------------- ----------------- 143.4/260.7 kB 448.2 kB/s eta 0:00:01
   -------------------- ----------------- 143.4/260.7 kB 448.2 kB/s eta 0:00:01
   ----------------------- -------------- 163.8/260.7 kB 409.6 kB/s eta 0:00:01
   ------------------------- ------------ 174.1/260.7 kB 374.1 kB/s eta 0:00:01
   ------------------------- ------------ 174.1/260.7 kB 374

In [3]:
import pandas as pd
import numpy as np
import librosa
import os
from sklearn.preprocessing import LabelEncoder

In [24]:
from pathlib import Path

cwd = Path.cwd()
path = cwd.parents[1] / "data" / "raw"  # up 2 levels, then /data/raw
processed_path = cwd.parents[1] / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)


In [25]:
ravdess_directory_list = os.listdir(path)
print(ravdess_directory_list)
file_emotion = []
file_path = []
for dir in ravdess_directory_list:
    actor = os.listdir(path / dir)
    for file in actor:
        part = file.split('.')[0]
        part = file.split('-')
        file_emotion.append(int(part[2]))
        file_path.append(str(path / dir / file))
        
emotion_df = pd.DataFrame(file_emotion, columns=['Emotions'])

path_df = pd.DataFrame(file_path, columns=['Path'])
meta_df = pd.concat([emotion_df, path_df], axis=1)

meta_df['Emotions'] = meta_df['Emotions'].replace({1:'neutral', 2:'calm', 3:'happy', 4:'sad', 5:'angry', 6:'fear', 7:'disgust', 8:'surprise'})

label_encoder = LabelEncoder()

meta_df["Emotion_ID"] = label_encoder.fit_transform(meta_df["Emotions"])

print(label_encoder.classes_)

meta_df.head()
meta_df.to_csv(processed_path / "ravdess_metadata.csv", index=False)

meta_df['Emotions'].value_counts()


['Actor_01', 'Actor_02', 'Actor_03', 'Actor_04', 'Actor_05', 'Actor_06', 'Actor_07', 'Actor_08', 'Actor_09', 'Actor_10', 'Actor_11', 'Actor_12', 'Actor_13', 'Actor_14', 'Actor_15', 'Actor_16', 'Actor_17', 'Actor_18', 'Actor_19', 'Actor_20', 'Actor_21', 'Actor_22', 'Actor_23', 'Actor_24']
['angry' 'calm' 'disgust' 'fear' 'happy' 'neutral' 'sad' 'surprise']


Emotions
calm        192
happy       192
sad         192
angry       192
fear        192
disgust     192
surprise    192
neutral      96
Name: count, dtype: int64

In [26]:
# Sample file to figure our sample rate and duration, from the output we know that the duration should be standardized and duration should be 48000 Hz
sample_path = meta_df["Path"].iloc[0]

audio, sr = librosa.load(sample_path, sr=None)

print("Sample rate:", sr)
print("Audio shape:", audio.shape)
print("Duration:", len(audio) / sr)

Sample rate: 48000
Audio shape: (158558,)
Duration: 3.3032916666666665


## Standardizing duration and extracting features

In [27]:
sr = 48000
duration = 3
length = sr * duration

def standardize_length(audio, target_length):
    # If the audio is longer than 3 seconds we cut it the middle bit, since often the emotion is most clear in the middle and there may be a delay at the start of the recording.
    # If its shorter we add silence at the end (represented by 0 in this case)
    if len(audio) > target_length:
        start = (len(audio) - target_length) // 2
        return audio[start:start + target_length]
    else:
        return np.pad(audio, (0, target_length - len(audio)))

In [28]:
processed_audio = []
labels_text = []
labels_num = []

for audio_path, emotion, emotion_id in zip(meta_df["Path"], meta_df["Emotions"], meta_df["Emotion_ID"]):
    # Load audio and resample to target sample rate
    audio, _ = librosa.load(audio_path, sr=sr)

    audio = standardize_length(audio, length)

    audio = librosa.util.normalize(audio) # Normalizes volume to avoid noise from recording levels etc.

    processed_audio.append(audio)
    labels_text.append(emotion)
    labels_num.append(emotion_id)

print("Processed clips:", len(processed_audio))
print("Text Labels:", len(labels_text))
print("Numeric Labels:", len(labels_num))

Processed clips: 1440
Text Labels: 1440
Numeric Labels: 1440


In [29]:
X_audio = np.array(processed_audio)
y_text = np.array(labels_text)
y_encoded = np.array(labels_num)

In [30]:
n_mfcc = 40 # Since the application for these features is emotion recognition, we went with 40 MFCCs to capture as much information as possible.

mfcc_features = []

for audio in X_audio:
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=n_mfcc
    )
    
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_features.append(mfcc_mean)

X_mfcc = np.array(mfcc_features)

print(X_mfcc.shape)

(1440, 40)


In [31]:
mfcc_df = pd.DataFrame(
    X_mfcc,
    columns=[f"mfcc_{i+1}" for i in range(n_mfcc)]
)

mfcc_df["Emotion"] = y_text
mfcc_df["Emotion_ID"] = y_encoded
mfcc_df["Path"] = meta_df["Path"].values

mfcc_df.head()

,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,...,mfcc_34,mfcc_35,mfcc_36,mfcc_37,mfcc_38,mfcc_39,mfcc_40,Emotion,Emotion_ID,Path
0,-398.042206,75.356049,3.555152,13.388279,6.033113,15.046699,-3.274679,3.410381,-3.609158,-1.716895,...,0.021143,-0.549320,-0.628053,0.052052,-1.343261,-1.760183,-1.589918,neutral,5,c:\Users\Kacpiro Recenzje\Desktop\Studia\XAI_2...
1,-404.414398,77.657196,1.182115,14.425774,8.691035,15.811256,-4.619638,4.848064,-4.008130,-4.158945,...,0.448995,-0.526972,-1.536152,0.292211,-1.084512,-2.310232,-1.154159,neutral,5,c:\Users\Kacpiro Recenzje\Desktop\Studia\XAI_2...
2,-423.030212,75.066849,3.681056,12.329444,6.456293,11.320028,-3.664177,4.332827,-5.347966,-3.426528,...,-0.462740,-0.964540,-1.042030,-0.227754,-1.294028,-1.938853,-1.648300,neutral,5,c:\Users\Kacpiro Recenzje\Desktop\Studia\XAI_2...
3,-431.444702,70.960457,5.918694,13.914023,6.412232,13.075801,-2.052994,5.827669,-5.217687,-3.630437,...,-0.344410,-0.806391,-0.634719,0.499814,-1.643585,-1.757895,-1.641027,neutral,5,c:\Users\Kacpiro Recenzje\Desktop\Studia\XAI_2...
4,-374.966949,88.756729,6.575492,16.645142,7.118174,16.783058,-3.964944,5.049708,-6.502436,-1.174364,...,0.530299,-1.210499,-1.660577,0.410300,-1.785656,-1.477737,-0.732335,calm,1,c:\Users\Kacpiro Recenzje\Desktop\Studia\XAI_2...


In [32]:
mfcc_df.to_csv(
    processed_path / "ravdess_mfcc_features.csv",
    index=False
)

## Additional preprocessing for CNN

In [33]:
n_mels = 128
# For Grad-cam we need to convert the .wav files into spectograms, since the model expects 2D input.
def extract_mel_spectrogram(audio, sr, n_mels=n_mels):
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels
    )
    
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    return mel_spec_db

In [34]:
mel_spectrograms = []

for audio in X_audio:
    mel_spec = extract_mel_spectrogram(audio, sr)
    mel_spectrograms.append(mel_spec)

X_mel = np.array(mel_spectrograms)

print("Mel spectrogram array shape:", X_mel.shape)

X_mel_cnn = X_mel[..., np.newaxis]

print("CNN input shape:", X_mel_cnn.shape)

Mel spectrogram array shape: (1440, 128, 282)
CNN input shape: (1440, 128, 282, 1)


In [35]:
np.save(processed_path / "ravdess_mel_spectrograms.npy", X_mel_cnn)
np.save(processed_path / "ravdess_emotion_labels_encoded.npy", y_encoded)
np.save(processed_path / "ravdess_emotion_classes.npy", label_encoder.classes_)

X_loaded = np.load(processed_path / "ravdess_mel_spectrograms.npy")
y_loaded = np.load(processed_path / "ravdess_emotion_labels_encoded.npy")

print("Loaded Mel spectrograms shape:", X_loaded.shape)
print("Loaded labels shape:", y_loaded.shape)

Loaded Mel spectrograms shape: (1440, 128, 282, 1)
Loaded labels shape: (1440,)
